## 08. Scaling

El escalado permite comparar predictores con magnitudes distintas y puede ser relevante para modelos que utilizan distancias, penalizaciones o gradientes. Los modelos basados en árboles habitualmente no necesitan el mismo tratamiento. El original registra candidatas con método model_dependent; no elige ni ajusta un scaler definitivo. Se conserva ese alcance.

In [ ]:
import pandas as pd
import numpy as np
from IPython.display import display

from house_prices_mlops.pipeline import load_stage
from house_prices_mlops.data_tracker import DataTracker

df_before = load_stage("common")
df_hp = df_before.copy()
tracker = DataTracker.load(df_before)
profile = tracker.profile
display(profile)


### Candidate Variables

Este notebook parte de common, sin ejecutar distribuciones, encoding ni selección de variables de otros notebooks. Cada experimento decidirá qué columnas conservar y escalar. Id y SalePrice no son candidatos a escalado de predictores.

In [ ]:
numeric_vars = df_hp.select_dtypes(include="number").columns.difference(["Id", "SalePrice"])
scaling_analysis = profile.loc[
    profile["var"].isin(numeric_vars),
    ["var", "variable_type", "unique_count", "min", "max"],
]
display(scaling_analysis)


### Derived Feature Classifications

El original clasifica explícitamente las edades, superficies agregadas e interacción como continuous, TotalBathrooms como discrete y tres indicadores como binary para estudiar scaling. Se conserva esta clasificación local de revisión; no se infiere a partir de dtypes ni se sobrescribe configs/variable_types.json. La aprobación y persistencia del catálogo central son una decisión separada.

In [ ]:
tracker.set_variable_types(
    [
        'HouseAge',
        'RemodAge',
        'GarageAge',
        'TotalSF',
        'TotalPorchSF',
        'QualGrLivArea'
    ],
    'continuous'
)

tracker.set_variable_types(
    [
        'TotalBathrooms'
    ],
    'discrete'
)

tracker.set_variable_types(
    [
        'HasBasement',
        'HasFireplace',
        'IsRemodeled'
    ],
    'binary'
)

In [ ]:
profile = tracker.profile
scaling_candidates = profile.loc[
    profile["variable_type"].eq("continuous")
    & ~profile["var"].isin(["Id", "SalePrice"]),
    "var",
].tolist()
display(profile.loc[profile["var"].isin(scaling_candidates), ["var", "variable_type", "min", "max"]])


### Scaling Strategies

- **StandardScaler:** centra usando la media y divide por la desviación estándar. Puede ser una alternativa para modelos lineales regularizados; es sensible a valores extremos.
- **RobustScaler:** utiliza mediana y un rango de cuantiles, habitualmente el IQR. Puede evaluarse cuando los extremos afectan mucho la escala; no elimina outliers.
- **MinMaxScaler:** utiliza los mínimos y máximos del entrenamiento para ajustar un rango. Los extremos influyen en ese rango y nuevos valores pueden quedar fuera de él.

Los tres aprenden estadísticas. Sus parámetros deben ajustarse dentro de cada fold del pipeline experimental y reutilizarse sin fit sobre validación y test. Scaling no garantiza normalidad ni sustituye a una imputación, transformación de distribución o selección. Todavía no se ejecuta ninguno aquí.

### Record Plans

In [ ]:
for var in scaling_candidates:
    tracker.add_plan(
        var=var,
        stage='DP - Scaling',
        action='scale',
        method='model_dependent',
        params={}
    )

In [ ]:
display(tracker.plans.loc[tracker.plans["stage"].eq("DP - Scaling")])


### Validation

In [ ]:
assert len(df_hp) == len(df_before)
pd.testing.assert_series_equal(df_hp["Id"], df_before["Id"])
pd.testing.assert_frame_equal(df_hp, df_before)
pd.testing.assert_frame_equal(df_before, load_stage("common"))
assert "Id" not in scaling_candidates
assert "SalePrice" not in scaling_candidates


### Save Decisions

In [ ]:
tracker.save()
